In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_3_9_17,0.998134,0.748276,0.998155,0.964250,0.987954,0.012480,1.683277,4.208691e-03,6.374373e-02,3.397618e-02,0.097969,0.111713,1.001317,0.116469,124.767288,195.462086,"Hidden Size=[7, 4], regularizer=0.3, learning_..."
1,model_3_9_18,0.998131,0.748254,0.998173,0.963086,0.987593,0.012500,1.683429,4.166592e-03,6.581978e-02,3.499318e-02,0.092567,0.111804,1.001320,0.116563,124.764046,195.458844,"Hidden Size=[7, 4], regularizer=0.3, learning_..."
2,model_3_9_16,0.998130,0.748297,0.998126,0.965524,0.988345,0.012504,1.683142,4.274667e-03,6.147276e-02,3.287372e-02,0.103971,0.111820,1.001320,0.116580,124.763466,195.458264,"Hidden Size=[7, 4], regularizer=0.3, learning_..."
3,model_3_9_19,0.998123,0.748230,0.998184,0.962023,0.987261,0.012552,1.683590,4.142083e-03,6.771538e-02,3.592873e-02,0.087708,0.112037,1.001325,0.116806,124.755721,195.450519,"Hidden Size=[7, 4], regularizer=0.3, learning_..."
4,model_3_9_15,0.998118,0.748313,0.998084,0.966913,0.988767,0.012588,1.683034,4.371278e-03,5.899531e-02,3.168330e-02,0.110641,0.112195,1.001329,0.116971,124.750082,195.444880,"Hidden Size=[7, 4], regularizer=0.3, learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1172,model_13_7_13,0.964458,0.696465,1.000000,1.000000,1.000000,0.237669,2.029741,5.640745e-12,3.389858e-12,4.424738e-12,0.366265,0.487513,1.020805,0.508267,132.873757,212.100685,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
1182,model_13_7_22,0.964458,0.696465,1.000000,1.000000,1.000000,0.237669,2.029741,5.640745e-12,3.389858e-12,4.424738e-12,0.366265,0.487513,1.020805,0.508267,132.873757,212.100685,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
1197,model_13_7_0,0.964458,0.696465,1.000000,1.000000,1.000000,0.237669,2.029741,5.640745e-12,3.389858e-12,4.424738e-12,0.366265,0.487513,1.020805,0.508267,132.873757,212.100685,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
1208,model_13_7_21,0.964458,0.696465,1.000000,1.000000,1.000000,0.237669,2.029741,5.640745e-12,3.389858e-12,4.424738e-12,0.366265,0.487513,1.020805,0.508267,132.873757,212.100685,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
